# Milestone 3 — Advanced EDA dan Analisis Corpus-Indonesia

**Dataset:** [Lyon28/Corpus-Indonesia](https://huggingface.co/datasets/Lyon28/Corpus-Indonesia)  
**Input:** `data/processed/corpus_indonesia_cleaned.parquet`  
**Kolom sumber:** `text`  
**Tools:** Polars, DuckDB, Plotly Express, Plotly Graph Objects

## Tujuan
1. Melanjutkan analisis dari dataset hasil cleaning Milestone 2.
2. Melakukan analisis distribusi panjang dokumen dan karakteristik teks.
3. Menerapkan operasi window/rolling pada urutan batch data.
4. Membuat minimal enam visualisasi interaktif.
5. Menyusun minimal lima insight berdasarkan hasil komputasi aktual.

## Catatan metodologis penting
Dataset ini hanya memiliki satu kolom, yaitu `text`. Tidak tersedia timestamp, tanggal publikasi, sumber berita, kategori berita, maupun koordinat geografis. Oleh sebab itu:
- Analisis rolling dilakukan terhadap **urutan batch/posisi record**, bukan tren kalender.
- Analisis spasial geografis tidak dapat dilakukan secara valid dari kolom yang tersedia.
- Analisis hubungan multi-dimensi menggunakan fitur turunan teks, seperti jumlah karakter dan jumlah kata.
- Statistik berbasis sampel diberi label sebagai estimasi sampel. Statistik agregat yang dihitung dari seluruh file akan disebut statistik seluruh dataset.

Jalankan notebook dari folder proyek `tugas1-corpus-indonesia` atau subfolder `notebooks`.


## 1. Inisialisasi dan Memuat Dataset Bersih

Notebook menggunakan LazyFrame agar seluruh corpus tidak perlu dimuat sekaligus ke RAM. Pastikan Milestone 2 telah menghasilkan file Parquet bersih.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import re
import warnings

import polars as pl
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=UserWarning)

# Cari root proyek dari working directory atau parent directory.
CWD = Path.cwd().resolve()
ROOT = next(
    (
        p for p in [CWD, *CWD.parents]
        if (p / "data" / "processed" / "corpus_indonesia_cleaned.parquet").is_file()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "File data/processed/corpus_indonesia_cleaned.parquet tidak ditemukan. "
        "Pastikan notebook berada di dalam proyek tugas1-corpus-indonesia "
        "dan Milestone 2 sudah dijalankan."
    )

CLEAN_PATH = ROOT / "data" / "processed" / "corpus_indonesia_cleaned.parquet"
OUT_TABLES = ROOT / "output" / "tables"
OUT_FIGURES = ROOT / "output" / "figures"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

lf = pl.scan_parquet(str(CLEAN_PATH))
schema = lf.collect_schema()

if "text" not in schema.names():
    raise ValueError(f"Kolom 'text' tidak ditemukan. Skema saat ini: {schema}")

print("Root proyek :", ROOT)
print("File input  :", CLEAN_PATH)
print("Polars      :", pl.__version__)
print("DuckDB      :", duckdb.__version__)
print("Plotly      :", px.__version__ if hasattr(px, "__version__") else "terpasang")
print("Skema       :", schema)


In [ ]:
# Hitung jumlah record aktual dari file bersih.
total_rows = int(
    lf.select(pl.len().alias("total_baris")).collect().item()
)

file_size_mb = CLEAN_PATH.stat().st_size / 1_000_000

print(f"Jumlah baris bersih : {total_rows:,}")
print(f"Ukuran file bersih  : {file_size_mb:,.2f} MB")
print(f"Jumlah kolom        : {len(schema)}")


## 2. Profiling Ulang Dataset Bersih

Statistik berikut dihitung dari seluruh record. Tujuannya memastikan input Milestone 3 konsisten dengan hasil validasi Milestone 2.


In [ ]:
text_length = pl.col("text").str.len_chars()
blank_expr = (
    pl.col("text").is_not_null()
    & (pl.col("text").str.strip_chars() == "")
)

profile = lf.select(
    pl.len().alias("total_baris"),
    pl.col("text").null_count().alias("text_null"),
    blank_expr.sum().alias("text_kosong_atau_spasi"),
    text_length.min().alias("panjang_min_karakter"),
    text_length.max().alias("panjang_maks_karakter"),
    text_length.mean().alias("panjang_rata_rata"),
    text_length.median().alias("panjang_median"),
    text_length.quantile(0.25).alias("q1_panjang"),
    text_length.quantile(0.75).alias("q3_panjang"),
).collect()

display(profile)

p = profile.row(0, named=True)
q1 = float(p["q1_panjang"])
q3 = float(p["q3_panjang"])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
lower_fence = max(q1 - 1.5 * iqr, 0)

print(f"Batas bawah IQR : {lower_fence:,.2f} karakter")
print(f"Batas atas IQR  : {upper_fence:,.2f} karakter")


In [ ]:
# Hitung jumlah dokumen di luar batas atas IQR pada seluruh dataset.
outlier_summary = lf.select(
    (text_length > upper_fence).sum().alias("dokumen_di_atas_batas_iqr"),
    (text_length < lower_fence).sum().alias("dokumen_di_bawah_batas_iqr"),
).collect()

display(outlier_summary)

outlier_count = int(outlier_summary["dokumen_di_atas_batas_iqr"][0])
outlier_pct = outlier_count / total_rows * 100 if total_rows else 0

print(f"Dokumen di atas batas IQR : {outlier_count:,} ({outlier_pct:.2f}%)")
print("Catatan: dokumen panjang tidak otomatis dianggap salah atau dihapus.")


## 3. Membentuk Sampel untuk Analisis Teks

Sebagian visualisasi dan analisis token menggunakan sampel sistematis agar proses tetap ringan. Sampel diambil dengan interval baris yang merata dari seluruh file, bukan hanya dari bagian awal corpus.


In [ ]:
SAMPLE_TARGET = 50_000
stride = max(total_rows // SAMPLE_TARGET, 1)

sample_lf = (
    lf.with_row_index("row_id")
    .filter((pl.col("row_id") % stride) == 0)
    .select("row_id", "text")
)

sample_df = sample_lf.collect(engine="streaming")

sample_df = sample_df.with_columns(
    pl.col("text").str.len_chars().alias("char_count"),
    pl.col("text").str.count_matches(r"\\S+").alias("word_count"),
)

sample_pd = sample_df.to_pandas()

print(f"Target sampel       : {SAMPLE_TARGET:,}")
print(f"Interval pengambilan: setiap {stride:,} record")
print(f"Jumlah sampel aktual: {sample_df.height:,}")
display(sample_df.head(10))


## 4. Rekayasa Fitur dan Kategori Panjang Teks

Fitur turunan dibuat dari satu-satunya kolom sumber, yaitu `text`. Kategori panjang teks dihitung pada seluruh dataset agar komposisi kategori tidak hanya mewakili sampel.


In [ ]:
length_category = (
    pl.when(text_length <= 50).then(pl.lit("01. 1–50 karakter"))
    .when(text_length <= 100).then(pl.lit("02. 51–100 karakter"))
    .when(text_length <= 250).then(pl.lit("03. 101–250 karakter"))
    .when(text_length <= 500).then(pl.lit("04. 251–500 karakter"))
    .when(text_length <= 1000).then(pl.lit("05. 501–1.000 karakter"))
    .otherwise(pl.lit("06. >1.000 karakter"))
)

category_df = (
    lf.with_columns(length_category.alias("kategori_panjang"))
    .group_by("kategori_panjang")
    .agg(pl.len().alias("jumlah_dokumen"))
    .sort("kategori_panjang")
    .collect()
)

category_pd = category_df.to_pandas()
category_pd["persentase"] = (
    category_pd["jumlah_dokumen"] / total_rows * 100
)

display(category_pd)


## 5. Analisis Window / Rolling Berdasarkan Urutan Batch

Corpus tidak memiliki timestamp. Untuk memenuhi penerapan operasi window secara jujur, record dibagi ke dalam batch berurutan. Rata-rata panjang teks tiap batch dihitung dari seluruh dataset, lalu dihitung rolling mean lima batch.

**Interpretasi:** sumbu X adalah urutan batch dalam file, bukan tanggal atau waktu publikasi. Pola ini hanya menggambarkan variasi menurut urutan penyimpanan data.


In [ ]:
BATCH_SIZE = 50_000

batch_summary = (
    lf.with_row_index("row_id")
    .with_columns(
        (pl.col("row_id") // BATCH_SIZE).alias("batch_id"),
        text_length.alias("char_count"),
    )
    .group_by("batch_id")
    .agg(
        pl.len().alias("jumlah_dokumen"),
        pl.col("char_count").mean().alias("rata_rata_karakter"),
        pl.col("char_count").median().alias("median_karakter"),
    )
    .sort("batch_id")
    .collect(engine="streaming")
    .with_columns(
        pl.col("rata_rata_karakter")
        .rolling_mean(window_size=5, min_samples=1)
        .alias("rolling_mean_5_batch"),
        pl.col("rata_rata_karakter")
        .shift(1)
        .alias("rata_rata_batch_sebelumnya"),
    )
)

batch_pd = batch_summary.to_pandas()
display(batch_summary.head(10))
print(f"Jumlah batch: {batch_summary.height:,}")


## 6. Analisis Frekuensi Kata

Frekuensi token dihitung dari sampel sistematis. Tokenisasi menggunakan ekspresi reguler Unicode sederhana. Daftar stopword di bawah digunakan untuk membantu membaca kata yang lebih informatif, bukan sebagai proses pembersihan permanen terhadap corpus.


In [ ]:
STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan",
    "pada", "adalah", "dalam", "tidak", "akan", "juga", "atau", "karena",
    "sebagai", "oleh", "dapat", "ada", "lebih", "sudah", "telah", "saat",
    "dalam", "agar", "bagi", "para", "kami", "kita", "mereka", "ia",
    "dia", "saya", "anda", "tersebut", "bisa", "masih", "hingga", "antara",
    "menjadi", "yakni", "yaitu", "serta", "secara", "setelah", "sebelum",
    "jika", "bukan", "pun", "hanya", "satu", "dua", "hal", "tahun"
}

token_counter = Counter()

for document in sample_df["text"].to_list():
    tokens = re.findall(r"[^\\W_]+", document.lower(), flags=re.UNICODE)
    token_counter.update(
        token for token in tokens
        if len(token) > 1 and token not in STOPWORDS_ID
    )

top_tokens = token_counter.most_common(20)
token_pd = pd.DataFrame(top_tokens, columns=["kata", "frekuensi_sampel"])

display(token_pd)
print("Catatan: frekuensi kata merupakan hasil estimasi dari sampel sistematis.")


## 7. Enam Visualisasi Interaktif

Semua grafik menggunakan Plotly dan disimpan sebagai file HTML interaktif pada folder `output/figures/`.


In [ ]:
# Visualisasi 1 — Rolling mean panjang teks per urutan batch.
fig1 = px.line(
    batch_pd,
    x="batch_id",
    y="rolling_mean_5_batch",
    markers=True,
    title="Visualisasi 1. Rolling Mean Panjang Teks per 5 Batch",
    labels={
        "batch_id": "Urutan batch (50.000 record)",
        "rolling_mean_5_batch": "Rolling mean panjang teks (karakter)"
    },
    hover_data=["jumlah_dokumen", "rata_rata_karakter"]
)
fig1.update_layout(hovermode="x unified")
fig1.show()
fig1.write_html(OUT_FIGURES / "01_rolling_mean_batch.html")


In [ ]:
# Visualisasi 2 — Distribusi panjang teks pada sampel sistematis.
fig2 = px.histogram(
    sample_pd,
    x="char_count",
    nbins=60,
    title="Visualisasi 2. Distribusi Panjang Teks",
    labels={"char_count": "Panjang teks (karakter)", "count": "Jumlah dokumen"},
    marginal="box"
)
fig2.show()
fig2.write_html(OUT_FIGURES / "02_distribusi_panjang_teks.html")


In [ ]:
# Visualisasi 3 — Box plot panjang teks untuk melihat sebaran dan pencilan.
fig3 = px.box(
    sample_pd,
    y="char_count",
    points="outliers",
    title="Visualisasi 3. Box Plot Panjang Teks pada Sampel",
    labels={"char_count": "Panjang teks (karakter)"}
)
fig3.update_yaxes(type="log")
fig3.show()
fig3.write_html(OUT_FIGURES / "03_boxplot_panjang_teks.html")


In [ ]:
# Visualisasi 4 — Komposisi kategori panjang teks dari seluruh dataset.
fig4 = px.bar(
    category_pd,
    x="kategori_panjang",
    y="jumlah_dokumen",
    text="persentase",
    title="Visualisasi 4. Komposisi Kategori Panjang Teks",
    labels={
        "kategori_panjang": "Kategori panjang teks",
        "jumlah_dokumen": "Jumlah dokumen",
        "persentase": "Persentase (%)"
    }
)
fig4.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig4.update_layout(xaxis_tickangle=-20)
fig4.show()
fig4.write_html(OUT_FIGURES / "04_komposisi_kategori_teks.html")


In [ ]:
# Visualisasi 5 — 20 kata paling sering muncul pada sampel setelah stopword sederhana dikeluarkan.
fig5 = px.bar(
    token_pd.sort_values("frekuensi_sampel", ascending=True),
    x="frekuensi_sampel",
    y="kata",
    orientation="h",
    title="Visualisasi 5. 20 Kata Paling Sering pada Sampel",
    labels={"frekuensi_sampel": "Frekuensi pada sampel", "kata": "Token"}
)
fig5.show()
fig5.write_html(OUT_FIGURES / "05_frekuensi_kata.html")


In [ ]:
# Visualisasi 6 — Hubungan jumlah karakter dan jumlah kata.
# Scatter menggunakan sebagian sampel untuk menjaga responsivitas browser.
scatter_pd = sample_pd.sample(
    n=min(10_000, len(sample_pd)),
    random_state=42
)

fig6 = px.scatter(
    scatter_pd,
    x="word_count",
    y="char_count",
    opacity=0.45,
    title="Visualisasi 6. Hubungan Jumlah Kata dan Jumlah Karakter",
    labels={
        "word_count": "Jumlah kata (estimasi)",
        "char_count": "Jumlah karakter"
    },
    hover_data=["row_id"]
)
fig6.show()
fig6.write_html(OUT_FIGURES / "06_hubungan_kata_karakter.html")


## 8. Ringkasan Insight Analitik

Lima insight berikut dihasilkan dari statistik aktual. Insight rolling tetap ditafsirkan sebagai urutan record, sedangkan frekuensi token berasal dari sampel sistematis.


In [ ]:
# Insight 1 — Profil umum dan kondisi kualitas data.
null_final = int(p["text_null"])
blank_final = int(p["text_kosong_atau_spasi"])
mean_len = float(p["panjang_rata_karakter"])
median_len = float(p["panjang_median"])

print("INSIGHT 1 — KUALITAS DAN PROFIL CORPUS")
print(f"Dataset berisi {total_rows:,} dokumen dengan satu variabel teks.")
print(f"Nilai null: {null_final:,}; teks kosong/spasi: {blank_final:,}.")
print(f"Rata-rata panjang teks {mean_len:.2f} karakter, sedangkan median {median_len:.2f} karakter.")
print("Perbedaan mean dan median dapat digunakan untuk menilai kemencengan distribusi panjang teks.")


In [ ]:
# Insight 2 — Sebaran panjang teks dan pencilan.
print("INSIGHT 2 — DISTRIBUSI DAN PENCILAN")
print(f"Q1: {q1:.2f}; Q3: {q3:.2f}; IQR: {iqr:.2f}.")
print(f"Batas atas IQR: {upper_fence:.2f} karakter.")
print(f"Dokumen di atas batas atas IQR: {outlier_count:,} ({outlier_pct:.2f}%).")
print("Panjang ekstrem perlu diperiksa berdasarkan konteks isi, bukan otomatis dihapus.")


In [ ]:
# Insight 3 — Kategori panjang teks yang dominan.
dominant_category = category_pd.sort_values(
    "jumlah_dokumen", ascending=False
).iloc[0]

print("INSIGHT 3 — KOMPOSISI PANJANG TEKS")
print(
    f"Kategori terbanyak adalah {dominant_category['kategori_panjang']} "
    f"dengan {int(dominant_category['jumlah_dokumen']):,} dokumen "
    f"({dominant_category['persentase']:.2f}% dari seluruh corpus)."
)
print("Kategori dominan menunjukkan rentang panjang dokumen yang paling banyak menyusun corpus.")


In [ ]:
# Insight 4 — Variasi panjang teks menurut urutan batch.
batch_min = float(batch_pd["rata_rata_karakter"].min())
batch_max = float(batch_pd["rata_rata_karakter"].max())
batch_min_id = int(batch_pd.loc[batch_pd["rata_rata_karakter"].idxmin(), "batch_id"])
batch_max_id = int(batch_pd.loc[batch_pd["rata_rata_karakter"].idxmax(), "batch_id"])

print("INSIGHT 4 — POLA BERDASARKAN URUTAN BATCH")
print(f"Rata-rata panjang batch terendah: {batch_min:.2f} karakter (batch {batch_min_id}).")
print(f"Rata-rata panjang batch tertinggi: {batch_max:.2f} karakter (batch {batch_max_id}).")
print("Pola ini menggambarkan variasi menurut urutan record, bukan tren waktu publikasi.")


In [ ]:
# Insight 5 — Pola leksikal dan rekomendasi.
if len(token_pd):
    first_token = token_pd.iloc[0]
    print("INSIGHT 5 — POLA KATA DAN REKOMENDASI")
    print(
        f"Token paling sering pada sampel setelah stopword sederhana dikeluarkan "
        f"adalah '{first_token['kata']}' dengan frekuensi {int(first_token['frekuensi_sampel']):,}."
    )
else:
    print("Tidak ada token yang memenuhi aturan tokenisasi pada sampel.")

print(
    "Rekomendasi: untuk analisis lanjutan, lakukan klasifikasi topik atau pemodelan bahasa "
    "setelah menentukan skema label dan evaluasi. Hindari menyimpulkan topik corpus hanya "
    "dari panjang teks atau frekuensi kata."
)


## 9. Ekspor Tabel Hasil dan Ringkasan

Tabel agregat dan sampel disimpan terpisah dari data mentah. Tidak ada perubahan terhadap file Parquet hasil cleaning.


In [ ]:
# Simpan tabel ringkasan yang dapat digunakan untuk laporan.
profile.write_csv(OUT_TABLES / "milestone3_profile_summary.csv")
category_df.write_csv(OUT_TABLES / "milestone3_text_length_categories.csv")
batch_summary.write_csv(OUT_TABLES / "milestone3_batch_rolling_summary.csv")
token_pd.to_csv(OUT_TABLES / "milestone3_top_tokens_sample.csv", index=False)

insight_summary = {
    "dataset": "Corpus-Indonesia",
    "input_file": "data/processed/corpus_indonesia_cleaned.parquet",
    "total_rows": total_rows,
    "columns": schema.names(),
    "sample_target": SAMPLE_TARGET,
    "sample_actual": int(sample_df.height),
    "sample_stride": int(stride),
    "batch_size": BATCH_SIZE,
    "batch_count": int(batch_summary.height),
    "window_operation": "rolling_mean(window_size=5) pada rata-rata panjang teks per batch",
    "temporal_limitation": (
        "Dataset tidak memiliki timestamp. Batch index hanya menunjukkan urutan record, "
        "bukan tanggal atau waktu publikasi."
    ),
    "spatial_limitation": (
        "Dataset tidak memiliki koordinat atau atribut lokasi geografis."
    ),
    "outlier_upper_iqr": upper_fence,
    "outlier_count_above_upper_iqr": outlier_count,
    "outlier_percentage": outlier_pct,
    "dominant_length_category": str(dominant_category["kategori_panjang"]),
    "top_sample_token": str(token_pd.iloc[0]["kata"]) if len(token_pd) else None,
    "figures": [
        "01_rolling_mean_batch.html",
        "02_distribusi_panjang_teks.html",
        "03_boxplot_panjang_teks.html",
        "04_komposisi_kategori_teks.html",
        "05_frekuensi_kata.html",
        "06_hubungan_kata_karakter.html"
    ]
}

with open(OUT_TABLES / "milestone3_insight_summary.json", "w", encoding="utf-8") as f:
    json.dump(insight_summary, f, ensure_ascii=False, indent=2)

print("Ekspor selesai.")
print("Folder tabel     :", OUT_TABLES)
print("Folder visualisasi:", OUT_FIGURES)
print("Jumlah grafik HTML:", len(list(OUT_FIGURES.glob("*.html"))))


## 10. Kesimpulan Milestone 3

Lengkapi kesimpulan berdasarkan hasil aktual setelah seluruh cell berhasil dijalankan.

1. **Kualitas dan karakteristik corpus:** jelaskan jumlah dokumen, kondisi null/kosong, rata-rata, median, serta rentang panjang teks.
2. **Distribusi dan pencilan:** jelaskan kategori panjang teks yang dominan dan proporsi dokumen di atas batas IQR.
3. **Pola urutan batch:** jelaskan perubahan rata-rata panjang teks antarbatch dan rolling mean lima batch. Tegaskan bahwa ini bukan tren kalender.
4. **Karakteristik leksikal:** bahas token yang sering muncul berdasarkan sampel, dengan menyebutkan bahwa hasilnya bukan hitungan seluruh corpus.
5. **Rekomendasi:** usulkan analisis lanjutan seperti klasifikasi topik, clustering, atau pemodelan bahasa dengan rancangan evaluasi yang sesuai.

### Batasan analisis
Dataset hanya menyediakan kolom `text`. Analisis temporal berbasis tanggal dan analisis spasial geografis tidak dapat disimpulkan dari data yang tersedia. Untuk melakukan analisis tersebut diperlukan metadata tambahan yang relevan.
